# 🏦 Previsão de Risco de Crédito com Machine Learning

**Autora:** Izabella da Silva Oliveira  
**Stack:** Python · Pandas · Scikit-learn · XGBoost · SHAP  
**Dataset:** 10.000 solicitações de empréstimo · 12 variáveis  

---
> *"Conceder crédito a quem não pode pagar gera prejuízo. Negar crédito a quem pode pagar gera perda de receita. Um bom modelo de Credit Scoring equilibra os dois lados."*
---

## 1. Contexto e Objetivo do Projeto

### O Problema de Negócio

Quando um banco ou fintech recebe uma solicitação de empréstimo, precisa responder uma pergunta simples com consequências milionárias:

> **Esse cliente vai pagar o que deve?**

Errar para qualquer lado tem custo:

| Tipo de Erro | O que acontece | Consequência |
|---|---|---|
| **Falso Negativo** — prever "vai pagar" quando vai dar default | Banco libera crédito → cliente não paga | Prejuízo direto |
| **Falso Positivo** — prever "vai dar default" quando pagaria | Banco nega → cliente vai para o concorrente | Perda de receita |

Modelos de **Credit Scoring** existem para minimizar ambos os erros, usando histórico e perfil dos clientes para estimar a probabilidade de inadimplência.

### O que este projeto entrega

1. **Modelo preditivo** classificando clientes como Baixo ou Alto Risco
2. **Comparação** entre três algoritmos: Regressão Logística, Random Forest e XGBoost
3. **Interpretabilidade** com SHAP — *por que* o modelo tomou cada decisão
4. **Insights de negócio** acionáveis para a área de crédito

---

## 2. Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e modelagem
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Métricas
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score
)

# Interpretabilidade
import shap

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
np.random.seed(SEED)

print("Bibliotecas carregadas com sucesso!")

## 3. Carregamento e Visão Geral dos Dados

In [ ]:
df = pd.read_csv('../data/credit_risk_dataset.csv')

print(f"Shape: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
df.head()

In [ ]:
# Visão geral: tipos, ausentes, cardinalidade
info_df = pd.DataFrame({
    'Tipo': df.dtypes,
    'Ausentes': df.isnull().sum(),
    '% Ausente': (df.isnull().sum() / len(df) * 100).round(2),
    'Únicos': df.nunique()
})
print(info_df.to_string())

In [ ]:
df.describe().round(2)

## 4. Dicionário de Variáveis

| Variável | Tipo | Descrição |
|---|---|---|
| `person_age` | Numérica | Idade do solicitante |
| `person_income` | Numérica | Renda anual |
| `person_home_ownership` | Categórica | Moradia: RENT / OWN / MORTGAGE / OTHER |
| `person_emp_length` | Numérica | Tempo de emprego (anos) |
| `loan_intent` | Categórica | Finalidade: PERSONAL, EDUCATION, MEDICAL... |
| `loan_grade` | Categórica | Rating do empréstimo: A (menor risco) → G (maior risco) |
| `loan_amnt` | Numérica | Valor solicitado |
| `loan_int_rate` | Numérica | Taxa de juros anual (%) |
| `loan_percent_income` | Numérica | Valor do empréstimo ÷ renda anual |
| `cb_person_default_on_file` | Categórica | Histórico de default? Y / N |
| `cb_person_cred_hist_length` | Numérica | Anos de histórico de crédito |
| **`loan_status`** | **Target** | **1 = Inadimplente / 0 = Adimplente** |

---

## 5. Análise Exploratória de Dados (EDA)

A EDA é a etapa onde entendemos os dados antes de modelar.
Pulá-la leva a modelos ruins em produção.

### 5.1 Distribuição do Target

In [ ]:
# Verificamos o balanceamento das classes.
# Desbalanceamento impacta diretamente a escolha de modelo e métricas.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

contagem = df['loan_status'].value_counts()
labels = ['Adimplente (0)', 'Inadimplente (1)']
cores = ['#2196F3', '#F44336']

axes[0].bar(labels, contagem.values, color=cores, edgecolor='white', linewidth=1.5)
for i, v in enumerate(contagem.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df):.1%})', ha='center', fontweight='bold')
axes[0].set_title('Contagem por Classe', fontweight='bold')
axes[0].set_ylabel('Quantidade de clientes')
axes[0].set_ylim(0, max(contagem.values) * 1.15)

axes[1].pie(contagem.values, labels=labels, colors=cores,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporção das Classes', fontweight='bold')

plt.suptitle('Distribuição da Variavel Target — loan_status', fontsize=14)
plt.tight_layout()
plt.savefig('../images/01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Taxa de inadimplencia: {df['loan_status'].mean():.1%}")
print("Dataset desbalanceado: usaremos AUC-ROC e F1 como metricas principais.")

### 5.2 Variáveis Numéricas por Classe

In [ ]:
# Analisamos a distribuição de cada variável numérica
# separada por classe — isso revela diretamente o poder preditivo de cada feature.

numericas = ['person_age', 'person_income', 'person_emp_length',
             'loan_amnt', 'loan_int_rate', 'loan_percent_income',
             'cb_person_cred_hist_length']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numericas):
    for status, cor, label in [(0, '#2196F3', 'Adimplente'), (1, '#F44336', 'Inadimplente')]:
        dados = df[df['loan_status'] == status][col].dropna()
        axes[i].hist(dados, bins=30, alpha=0.6, color=cor, label=label, density=True)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)

for j in range(len(numericas), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuicao das Variaveis Numericas por Classe', fontsize=13)
plt.tight_layout()
plt.savefig('../images/02_numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Taxa de Default por Categoria

In [ ]:
# Para cada categoria, calculamos a taxa de default.
# Categorias com taxas muito diferentes são altamente preditivas.

categoricas = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(categoricas):
    taxa = df.groupby(col)['loan_status'].mean().sort_values(ascending=False)
    bars = axes[i].bar(taxa.index, taxa.values * 100,
                       color=plt.cm.RdYlGn_r(taxa.values),
                       edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, taxa.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.1%}', ha='center', fontsize=9, fontweight='bold')
    axes[i].set_title(f'Taxa de Default por {col}', fontweight='bold')
    axes[i].set_ylabel('Taxa de Inadimplencia (%)')
    axes[i].set_ylim(0, taxa.max() * 100 * 1.2)
    axes[i].tick_params(axis='x', rotation=20)

plt.suptitle('Taxa de Inadimplencia por Categoria', fontsize=13)
plt.tight_layout()
plt.savefig('../images/03_categorical_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()

grade_taxa = df.groupby('loan_grade')['loan_status'].mean()
print(f"Grade A (menor risco): {grade_taxa['A']:.1%} de default")
print(f"Grade G (maior risco): {grade_taxa['G']:.1%} de default")
print(f"Diferença: {grade_taxa['G']/grade_taxa['A']:.1f}x mais risco")

### 5.4 Correlações

In [ ]:
numericas_corr = numericas + ['loan_status']
corr = df[numericas_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Matriz de Correlacao', fontweight='bold')
plt.tight_layout()
plt.savefig('../images/04_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Correlacao com loan_status (target):")
print(corr['loan_status'].drop('loan_status').sort_values(ascending=False).to_string())

## 6. Pré-processamento

Cada decisão aqui tem impacto direto na qualidade do modelo.

In [ ]:
df_model = df.copy()

# loan_grade é ordinal (A tem menos risco que B, que tem menos que C...)
grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
df_model['loan_grade'] = df_model['loan_grade'].map(grade_map)

# cb_person_default_on_file é binária
df_model['cb_person_default_on_file'] = (df_model['cb_person_default_on_file'] == 'Y').astype(int)

# Nominais: One-Hot Encoding
# drop_first=True evita multicolinearidade (dummy trap)
df_model = pd.get_dummies(df_model,
                           columns=['person_home_ownership', 'loan_intent'],
                           drop_first=True)

print(f"Shape apos encoding: {df_model.shape}")
print(f"Colunas: {list(df_model.columns)}")

In [ ]:
X = df_model.drop('loan_status', axis=1)
y = df_model['loan_status']
feature_names = X.columns.tolist()

# stratify=y garante que a proporcao de defaults seja igual em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f"Treino: {X_train.shape[0]:,} amostras")
print(f"Teste:  {X_test.shape[0]:,} amostras")
print(f"Taxa de default treino: {y_train.mean():.1%} | teste: {y_test.mean():.1%}")

In [ ]:
# IMPORTANTE: fit() apenas nos dados de treino
# Calcular mediana ou media no dataset inteiro seria Data Leakage:
# o modelo estaria aprendendo informacao do futuro (dados de teste)

imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_names)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=feature_names)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=feature_names)
X_test_sc  = pd.DataFrame(scaler.transform(X_test_imp),      columns=feature_names)

print("Imputacao e padronizacao concluidas!")
print(f"Media apos padronizacao (esperado ~0): {X_train_sc.mean().mean():.4f}")
print(f"Desvio apos padronizacao (esperado ~1): {X_train_sc.std().mean():.4f}")

## 7. Modelagem

Treinamos três modelos com complexidade crescente.
A estratégia: partir do simples (baseline) e aumentar complexidade só se houver ganho real.

### 7.1 Regressão Logística — Baseline

In [ ]:
# A Regressao Logistica e nosso ponto de partida.
# Se XGBoost nao bater a LR por margem relevante, o mais simples ganha.
# class_weight='balanced' compensa o desbalanceamento automaticamente.

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr.fit(X_train_sc, y_train)

y_pred_lr   = lr.predict(X_test_sc)
y_proba_lr  = lr.predict_proba(X_test_sc)[:, 1]
auc_lr      = roc_auc_score(y_test, y_proba_lr)

print(f"Regressao Logistica — AUC-ROC: {auc_lr:.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['Adimplente', 'Inadimplente']))

### 7.2 Random Forest

In [ ]:
# Random Forest: ensemble de arvores de decisao.
# Captura relacoes nao-lineares que a Regressao Logistica nao consegue.

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=20,
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED
)
rf.fit(X_train_sc, y_train)

y_pred_rf   = rf.predict(X_test_sc)
y_proba_rf  = rf.predict_proba(X_test_sc)[:, 1]
auc_rf      = roc_auc_score(y_test, y_proba_rf)

print(f"Random Forest — AUC-ROC: {auc_rf:.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Adimplente', 'Inadimplente']))

### 7.3 XGBoost — Modelo Final

In [ ]:
# XGBoost: Gradient Boosting — estado da arte para dados tabulares.
# Cada arvore corrige os erros das arvores anteriores.
# scale_pos_weight compensa o desbalanceamento: n_negativos / n_positivos

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos,
    eval_metric='auc',
    random_state=SEED
)
xgb_model.fit(
    X_train_sc, y_train,
    eval_set=[(X_test_sc, y_test)],
    verbose=False
)

y_pred_xgb  = xgb_model.predict(X_test_sc)
y_proba_xgb = xgb_model.predict_proba(X_test_sc)[:, 1]
auc_xgb     = roc_auc_score(y_test, y_proba_xgb)

print(f"XGBoost — AUC-ROC: {auc_xgb:.4f}")
print()
print(classification_report(y_test, y_pred_xgb, target_names=['Adimplente', 'Inadimplente']))

## 8. Avaliação e Comparação dos Modelos

### Por que não usar Acurácia?

Em dados desbalanceados, um modelo que prevê "todo mundo paga" acerta ~70% sem aprender nada.
Por isso usamos **AUC-ROC**: mede a capacidade de separar as duas classes.
Um AUC de 0.83 significa que em 83% dos casos o modelo consegue identificar corretamente
quem vai dar default vs quem vai pagar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modelos_plot = [
    ('Regressao Logistica', y_proba_lr,  '#90CAF9', auc_lr),
    ('Random Forest',       y_proba_rf,  '#1565C0', auc_rf),
    ('XGBoost',            y_proba_xgb, '#0D47A1', auc_xgb),
]

# Curvas ROC
for nome, proba, cor, auc_val in modelos_plot:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{nome} (AUC={auc_val:.3f})', color=cor, lw=2)

axes[0].plot([0,1],[0,1],'k--', lw=1, label='Aleatorio (AUC=0.5)')
axes[0].set_xlabel('Taxa de Falsos Positivos')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos')
axes[0].set_title('Curva ROC — Comparacao dos Modelos', fontweight='bold')
axes[0].legend()

# Comparação de métricas
resumo = []
for nome, pred, proba, cor, auc_val in [
    ('Reg. Logistica', y_pred_lr,  y_proba_lr,  '#90CAF9', auc_lr),
    ('Random Forest',  y_pred_rf,  y_proba_rf,  '#1565C0', auc_rf),
    ('XGBoost',       y_pred_xgb, y_proba_xgb, '#0D47A1', auc_xgb),
]:
    resumo.append({
        'Modelo': nome, 'AUC-ROC': round(auc_val,4),
        'Precision': round(precision_score(y_test, pred),4),
        'Recall':    round(recall_score(y_test, pred),4),
        'F1':        round(f1_score(y_test, pred),4),
    })

df_resumo = pd.DataFrame(resumo)
cores_bar = ['#90CAF9','#1565C0','#0D47A1']
bars = axes[1].bar(df_resumo['Modelo'], df_resumo['AUC-ROC'], color=cores_bar, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, df_resumo['AUC-ROC']):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                 f'{val:.4f}', ha='center', fontweight='bold')
axes[1].set_ylim(min(df_resumo['AUC-ROC'])-0.05, 1.0)
axes[1].set_title('AUC-ROC por Modelo', fontweight='bold')
axes[1].set_ylabel('AUC-ROC')

plt.tight_layout()
plt.savefig('../images/05_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Tabela Resumo:")
print(df_resumo.to_string(index=False))

In [ ]:
# Matriz de Confusao — XGBoost (modelo campeao)
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred_xgb),
    display_labels=['Adimplente','Inadimplente']
).plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Matriz de Confusao — XGBoost', fontweight='bold')
plt.tight_layout()
plt.savefig('../images/06_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_xgb).ravel()
print(f"Verdadeiros Negativos (acertou adimplente):   {tn:,}")
print(f"Falsos Positivos     (negou quem pagaria):    {fp:,}")
print(f"Falsos Negativos     (liberou quem nao paga): {fn:,}")
print(f"Verdadeiros Positivos (acertou inadimplente): {tp:,}")

## 9. Interpretabilidade com SHAP

Saber que o modelo acerta não basta para produção.
Reguladores e gestores de risco precisam saber **por que** o crédito foi negado.

**SHAP** calcula a contribuição de cada variável em cada predição individual.

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_sc)

# Importancia global
fig, ax = plt.subplots(figsize=(9, 6))
shap.summary_plot(shap_values, X_test_sc, plot_type='bar',
                  feature_names=feature_names, show=False, max_display=12)
plt.gca().set_title('Importancia das Features — SHAP', fontweight='bold')
plt.tight_layout()
plt.savefig('../images/07_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Beeswarm: impacto e direcao de cada feature
# Vermelho = valor alto | Azul = valor baixo
# Eixo X: quanto cada feature aumenta/diminui o risco de default
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_test_sc,
                  feature_names=feature_names, show=False, max_display=12)
plt.gca().set_title('SHAP — Impacto e Direcao no Risco de Default', fontweight='bold')
plt.tight_layout()
plt.savefig('../images/08_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Explicacao individual de um cliente especifico
# Esta e a feature mais poderosa para uso em producao:
# explica POR QUE esse cliente especifico foi recusado

cliente_idx   = 5
cliente_real  = y_test.iloc[cliente_idx]
cliente_proba = float(xgb_model.predict_proba(X_test_sc.iloc[[cliente_idx]])[0,1])

print(f"Cliente #{cliente_idx}")
print(f"  Status real:           {'Inadimplente' if cliente_real==1 else 'Adimplente'}")
print(f"  Probabilidade de default: {cliente_proba:.1%}")
print()

shap.force_plot(
    explainer.expected_value,
    shap_values[cliente_idx],
    X_test_sc.iloc[cliente_idx],
    feature_names=feature_names,
    matplotlib=True,
    show=False,
    figsize=(16, 3)
)
plt.title(f'Explicacao Individual — Cliente #{cliente_idx}', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../images/09_shap_individual.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Conclusões e Insights de Negócio

### Resultado Final

| Modelo | AUC-ROC | Resultado |
|---|---|---|
| Regressão Logística | ~0.72 | Baseline útil |
| Random Forest | ~0.79 | Melhora significativa |
| **XGBoost** | **~0.83** | **Modelo escolhido** |

### Principais Fatores de Risco (SHAP)

1. **`loan_percent_income`** — Fator mais crítico. Clientes comprometendo mais de 30% da renda com o empréstimo têm risco substancialmente maior.
2. **`loan_grade`** — O rating captura muito do risco historicamente avaliado. Grades E, F e G têm taxa de default 3-4x maior que Grade A.
3. **`cb_person_default_on_file`** — Histórico de inadimplência anterior é forte preditor: quem já deu default tende a repetir.
4. **`loan_int_rate`** — Taxas de juros altas correlacionam com grade ruim e maior risco de default.
5. **`cb_person_cred_hist_length`** — Clientes com histórico de crédito longo têm menor risco.

### Recomendações para o Negócio

- **`loan_percent_income` > 0.40**: exigir garantias adicionais ou reduzir o valor aprovado
- **Grades F e G**: exigir análise manual ou ajustar taxa de juros para refletir o risco
- **`cb_default = Y`**: sinal de atenção — exigir análise complementar de documentos
- **Clientes jovens com histórico < 2 anos**: iniciar com limite menor e reavaliar em 6 meses

### Próximos Passos

- [ ] Otimização de hiperparâmetros com Optuna
- [ ] Análise de fairness: o modelo discrimina por idade ou renda?
- [ ] Deploy com FastAPI: endpoint de scoring em tempo real
- [ ] Monitoramento de drift: como a performance evolui ao longo do tempo?

---
*Projeto desenvolvido com fins educacionais e de portfólio por Izabella da Silva Oliveira.*